# 02 — Medallion Pipeline: Bronze → Silver → Gold

**Purpose:** Transform the 5 raw Bronze tables into analysis-ready Silver
and Gold tables inside the same Unity Catalog schema.

**Catalog / Schema:** `dev.safety_stock_gold`

**Tables produced:**
| Table | Layer | Description |
|---|---|---|
| `demand_weekly` | Silver | IQR-cleaned, weekly-aggregated demand |
| `lead_times_cleaned` | Silver | Z-score-filtered lead times |
| `safety_stock_features` | Gold | ML-ready feature table (one row per material) |

**Prerequisite:** Run `01_create_dummy_data` first.

## 0. Configuration — Edit before running

In [ ]:
CATALOG = 'dev'
SCHEMA  = 'safety_stock_gold'

spark.sql(f'USE CATALOG {CATALOG}')
spark.sql(f'USE SCHEMA {SCHEMA}')
print(f'\u2705 Using {CATALOG}.{SCHEMA}')

Target: dev.safety_stock_gold


## 1. Imports

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import DoubleType, IntegerType

print('\u2705 Imports ready')

✅ Imports ready


## 2. Bronze → Silver: Demand Cleaning

Steps:
1. Remove daily demand outliers per material using the IQR method (1.5 × IQR fence)
2. Aggregate cleaned daily demand to **weekly** totals

In [ ]:
demand_raw = spark.table(f'{CATALOG}.{SCHEMA}.historical_demand')
print(f'Raw demand rows : {demand_raw.count():,}')

# ── IQR outlier fences per material-plant ────────────────────────────────────
fences = demand_raw.groupBy('material_id', 'plant').agg(
    F.percentile_approx('demand_qty', 0.25).alias('q1'),
    F.percentile_approx('demand_qty', 0.75).alias('q3'),
)
fences = (
    fences
    .withColumn('iqr',   F.col('q3') - F.col('q1'))
    .withColumn('lower', F.col('q1') - 1.5 * F.col('iqr'))
    .withColumn('upper', F.col('q3') + 1.5 * F.col('iqr'))
)

demand_filtered = (
    demand_raw
    .join(fences.select('material_id', 'plant', 'lower', 'upper'),
          on=['material_id', 'plant'], how='left')
    .filter(
        (F.col('demand_qty') >= F.col('lower')) &
        (F.col('demand_qty') <= F.col('upper'))
    )
    .select(demand_raw.columns)
)
print(f'After IQR filter: {demand_filtered.count():,}')

# ── Weekly aggregation ───────────────────────────────────────────────────────
demand_weekly = (
    demand_filtered
    .withColumn('week_start', F.date_trunc('week', F.col('date')))
    .groupBy('material_id', 'plant', 'week_start')
    .agg(F.sum('demand_qty').alias('weekly_demand'))
    .orderBy('material_id', 'week_start')
)
print(f'Weekly demand rows: {demand_weekly.count():,}')

Raw demand rows : 73,000
After IQR filter: 70,124
Weekly demand rows: 10,400


## 3. Bronze → Silver: Lead Time Cleaning

Remove per-material outliers where the lead time deviates by more than 3 standard
deviations from the material's mean (Z-score filter).

In [ ]:
lt_raw = spark.table(f'{CATALOG}.{SCHEMA}.lead_times')
print(f'Raw lead time rows    : {lt_raw.count():,}')

lt_stats = lt_raw.groupBy('material_id', 'plant').agg(
    F.mean('lead_time_days').alias('lt_mean'),
    F.stddev('lead_time_days').alias('lt_std'),
)

lt_cleaned = (
    lt_raw
    .join(lt_stats, on=['material_id', 'plant'], how='left')
    .filter(
        F.abs(F.col('lead_time_days') - F.col('lt_mean')) <= 3 * F.col('lt_std')
    )
    .select(lt_raw.columns)
)
print(f'After Z-score filter  : {lt_cleaned.count():,}')

Raw lead time rows    : 1,218
After Z-score filter  : 1,204


## 4. Silver → Gold: Feature Engineering

Compute one feature row per material-plant by aggregating the Silver tables
and joining with material master, buyer map, and current safety stock.

| Feature | Formula |
|---|---|
| `demand_mean` | Mean weekly demand |
| `demand_std` | Std dev of weekly demand |
| `demand_cv` | `demand_std / demand_mean` (coefficient of variation) |
| `lead_time_mean` | Mean lead time (days) |
| `lead_time_std` | Std dev of lead time (days) |
| `service_level_z` | Z-score mapped from service_level_target |
| `abc_class_encoded` | A→2, B→1, C→0 |
| `category_encoded` | Integer label per category |

In [ ]:
# ── Demand features ──────────────────────────────────────────────────────────
demand_feats = demand_weekly.groupBy('material_id', 'plant').agg(
    F.mean('weekly_demand').alias('demand_mean'),
    F.stddev('weekly_demand').alias('demand_std'),
    F.min('weekly_demand').alias('demand_min'),
    F.max('weekly_demand').alias('demand_max'),
    F.count('weekly_demand').alias('n_weeks'),
)
demand_feats = demand_feats.withColumn(
    'demand_cv',
    F.when(F.col('demand_mean') > 0, F.col('demand_std') / F.col('demand_mean')).otherwise(F.lit(0.0))
)

# ── Lead time features ───────────────────────────────────────────────────────
lt_feats = lt_cleaned.groupBy('material_id', 'plant').agg(
    F.mean('lead_time_days').alias('lead_time_mean'),
    F.stddev('lead_time_days').alias('lead_time_std'),
    F.min('lead_time_days').alias('lead_time_min'),
    F.max('lead_time_days').alias('lead_time_max'),
    F.count('lead_time_days').alias('n_pos'),
)

# ── Lookup tables ────────────────────────────────────────────────────────────
materials   = spark.table(f'{CATALOG}.{SCHEMA}.materials')
current_ss  = spark.table(f'{CATALOG}.{SCHEMA}.current_safety_stock')
buyers_raw  = spark.table(f'{CATALOG}.{SCHEMA}.buyers')

# Explode comma-separated material_ids in buyers
buyer_map = (
    buyers_raw
    .select('buyer_id', F.explode(F.split(F.col('material_ids'), ',')).alias('material_id'))
    .withColumn('material_id', F.trim(F.col('material_id')))
)

# Encoding lookup tables
z_map = spark.createDataFrame(
    [(0.90, 1.282), (0.95, 1.645), (0.99, 2.326)],
    ['service_level_target', 'service_level_z']
)
abc_map = spark.createDataFrame(
    [('A', 2), ('B', 1), ('C', 0)],
    ['abc_class', 'abc_class_encoded']
)
cat_map = spark.createDataFrame(
    [('Generators', 0), ('Engines', 1), ('Electrical', 2), ('Controls', 3), ('Accessories', 4)],
    ['category', 'category_encoded']
)

# ── Join everything ───────────────────────────────────────────────────────────
gold = (
    demand_feats
    .join(lt_feats, on=['material_id', 'plant'], how='left')
    .join(materials.select('material_id', 'plant', 'category', 'abc_class',
                           'service_level_target', 'material_desc'),
          on=['material_id', 'plant'], how='left')
    .join(current_ss.select('material_id', 'plant', 'current_ss'),
          on=['material_id', 'plant'], how='left')
    .join(buyer_map, on='material_id', how='left')
    .join(z_map, on='service_level_target', how='left')
    .join(abc_map, on='abc_class', how='left')
    .join(cat_map, on='category', how='left')
)

# Fill nulls from sparse joins
gold = gold.fillna({
    'lead_time_mean':    14.0,
    'lead_time_std':      3.0,
    'current_ss':        10,
    'demand_std':         0.0,
    'demand_cv':          0.0,
    'service_level_z':    1.645,
    'abc_class_encoded':  1,
    'category_encoded':   0,
})

gold = gold.withColumn('feature_computed_at', F.current_timestamp())

print(f'Gold feature rows: {gold.count():,}')

Gold feature rows: 100


## 5. Write Silver and Gold Tables

In [ ]:
def write_table(sdf, table_name: str, comment: str) -> None:
    (
        sdf.write
           .format('delta')
           .mode('overwrite')
           .option('overwriteSchema', 'true')
           .saveAsTable(f'{CATALOG}.{SCHEMA}.{table_name}')
    )
    safe_comment = comment.replace("'", "''")
    spark.sql(f"COMMENT ON TABLE {CATALOG}.{SCHEMA}.{table_name} IS '{safe_comment}'")
    count = spark.table(f'{CATALOG}.{SCHEMA}.{table_name}').count()
    print(f'  \u2705  {CATALOG}.{SCHEMA}.{table_name}  \u2192  {count:,} rows')

print('Writing tables...')

write_table(
    demand_weekly, 'demand_weekly',
    'Silver layer: IQR-cleaned daily demand aggregated to weekly totals per material-plant. '
    'Used for demand feature engineering in the safety stock ML model.'
)

write_table(
    lt_cleaned, 'lead_times_cleaned',
    'Silver layer: Z-score-filtered lead times per material-plant-vendor (outliers > 3 sigma removed). '
    'Used for lead time feature engineering in the safety stock ML model.'
)

write_table(
    gold, 'safety_stock_features',
    'Gold layer: ML-ready feature table with one row per material-plant. '
    'Contains demand statistics, lead time statistics, service level Z-score, '
    'encoded categorical features, current safety stock, and buyer assignment. '
    'Primary input to the safety stock ML model.'
)

print('\n\U0001F389 All medallion tables written successfully.')

Writing tables...
  ✅  dev.safety_stock_gold.demand_weekly  →  10,400 rows
  ✅  dev.safety_stock_gold.lead_times_cleaned  →  1,204 rows
  ✅  dev.safety_stock_gold.safety_stock_features  →  100 rows

🎉 All medallion tables written successfully.


## 6. Verify — Row Counts and Schema

In [ ]:
tables = ['demand_weekly', 'lead_times_cleaned', 'safety_stock_features']

print(f"{'Table':<35} {'Row count':>10}")
print('-' * 47)
for t in tables:
    count = spark.table(f'{CATALOG}.{SCHEMA}.{t}').count()
    print(f'{t:<35} {count:>10,}')

Table                              Row count
-------------------------------------------------
demand_weekly                         10,400
lead_times_cleaned                     1,204
safety_stock_features                    100


### Sample rows from `safety_stock_features`

In [ ]:
display(
    spark.table(f'{CATALOG}.{SCHEMA}.safety_stock_features')
        .select('material_id', 'abc_class', 'demand_mean', 'demand_cv',
                'lead_time_mean', 'lead_time_std', 'current_ss', 'buyer_id')
        .orderBy('abc_class', 'material_id')
        .limit(10)
)

## 7. Add Column-Level Comments

In [ ]:
col_comments = {
    'demand_weekly': {
        'material_id':   'Material identifier. Foreign key to materials.material_id.',
        'plant':         'Plant code.',
        'week_start':    'Start date of the ISO week (Monday).',
        'weekly_demand': 'Total units demanded in the week after IQR outlier removal.',
    },
    'lead_times_cleaned': {
        'material_id':    'Material identifier.',
        'plant':          'Plant code.',
        'po_date':        'Date the purchase order was placed.',
        'lead_time_days': 'Actual lead time in calendar days (outliers beyond 3 sigma removed).',
        'vendor_id':      'Vendor identifier.',
    },
    'safety_stock_features': {
        'material_id':        'Material identifier. Primary key for ML scoring.',
        'plant':              'Plant code.',
        'buyer_id':           'Buyer responsible for this material.',
        'demand_mean':        'Mean weekly demand (units/week) over the 2-year history.',
        'demand_std':         'Standard deviation of weekly demand.',
        'demand_cv':          'Coefficient of variation = demand_std / demand_mean. Higher value = more variable demand.',
        'lead_time_mean':     'Mean supplier lead time in days.',
        'lead_time_std':      'Standard deviation of lead time in days.',
        'current_ss':         'Current safety stock in SAP. Baseline for ML comparison.',
        'service_level_z':    'Z-score corresponding to the material target service level.',
        'abc_class_encoded':  'Numeric encoding of ABC class: A=2, B=1, C=0.',
        'category_encoded':   'Numeric label for product category (0=Generators ... 4=Accessories).',
    },
}

for table, cols in col_comments.items():
    for col, comment in cols.items():
        safe = comment.replace("'", "''")
        spark.sql(f'ALTER TABLE {CATALOG}.{SCHEMA}.{table} ALTER COLUMN {col} COMMENT \'{safe}\'')
    print(f'  \u2705  Column comments added \u2192 {table}')

print('\n\u2705 All column-level comments applied.')

  ✅  Column comments added → demand_weekly
  ✅  Column comments added → lead_times_cleaned
  ✅  Column comments added → safety_stock_features

✅ All column-level comments applied.


## Next Step

Run **`03_train_model`** to train the RandomForest safety stock model
on `safety_stock_features` and register it in MLflow Model Registry.